In [ ]:
pip install gwaslab 

In [56]:
import gwaslab as gl
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import gzip

In [57]:
import pandas as pd
import gzip

file_path = "../../data/susie/gwas/21001_raw.gwas.imputed_v3.both_sexes.tsv.bgz"
with gzip.open(file_path, 'rt') as f:
    gwas_data_df = pd.read_csv(f, sep='\t') 


In [58]:
gwas_data_df.head()

,variant,minor_allele,minor_AF,low_confidence_variant,n_complete_samples,AC,ytx,beta,se,tstat,pval
0,1:15791:C:T,T,5.446880e-09,True,359983,0.003922,1.238950e-01,894.616000,1204.870000,0.742499,0.457786
1,1:69487:G:A,A,5.768250e-06,True,359983,4.152940,1.035150e+02,-2.715450,2.360060,-1.150590,0.249902
2,1:69569:T:C,C,1.878960e-04,True,359983,135.278000,3.644560e+03,-0.484284,0.423462,-1.143630,0.252778
3,1:139853:C:T,T,5.681100e-06,True,359983,4.090200,1.018400e+02,-2.703560,2.360130,-1.145510,0.251997
4,1:692794:CA:C,C,1.105900e-01,False,359983,79621.100000,2.179000e+06,-0.016436,0.019585,-0.839228,0.401342


In [59]:
gwas_data_df[gwas_data_df["variant"]=="1:753405:C:A"]

,variant,minor_allele,minor_AF,low_confidence_variant,n_complete_samples,AC,ytx,beta,se,tstat,pval
20,1:753405:C:A,C,0.129036,False,359983,627065.0,17173900.0,0.000436,0.016685,0.026138,0.979147


In [60]:
gwas_data_df.head()

split_variant = gwas_data_df['variant'].str.split(':', expand=True)
gwas_data_df['CHR'] = split_variant[0]
gwas_data_df['POS'] = split_variant[1]
gwas_data_df['A2'] = split_variant[2]
gwas_data_df['A1'] = split_variant[3]
gwas_data_df.head()
gwas_data_df = gwas_data_df[~gwas_data_df["low_confidence_variant"]]
gwas_data_df = gwas_data_df[gwas_data_df["minor_AF"] > 0.01]
gwas_data_df = gwas_data_df.dropna(subset=["A1", "A2"])



In [62]:
gwas_data_df=gwas_data_df.rename(columns={'variant':'SNPID'})
gwas_data_df["SNPID"].to_csv("../../data/susie/mismatch_test/gwas_formatted_SNPs.txt", index=False)

In [63]:
gwas_data_df.to_csv("../../data/susie/mismatch_test/gwas_ready_for_harmo.csv", index=False)

In [64]:

ref_bim = pd.read_csv(
    "../../data/susie/plink_binary/merged_data.bim", 
    sep="\t", 
    header=None, 
    names=["CHR", "SNP", "CM", "POS", "A2", "A1"]
)


ref_bim["CHR"] = ref_bim["CHR"].astype(str)  
ref_bim["POS"] = ref_bim["POS"].astype(int)   

gwas_data_df["CHR"] = gwas_data_df["CHR"].astype(str)  
gwas_data_df["POS"] = gwas_data_df["POS"].astype(int)  


merged = pd.merge(
    gwas_data_df,
    ref_bim,
    on=["CHR", "POS"],
    suffixes=("_gwas", "_ref")
)


mismatch = (
    (merged["A1_gwas"] != merged["A1_ref"]) |
    (merged["A2_gwas"] != merged["A2_ref"])
)
print(f"Found {mismatch.sum()} allele mismatches.")

Found 4926933 allele mismatches.


#### sample SNPs for comparation 

#### For 1:753405:C:A  (not flipped)

From the vcf file format 
 CHROM    POS       ID        REF     ALT     QUAL    FILTER  INFO    FORMAT 

From the vcf file 

1       753405      .          C        A  

From the updated_vcf file 

1       753405  1:753405:C:A    C        A 

From row gwas 

variant	minor_allele	minor_AF	low_confidence_variant	n_complete_samples	AC	ytx	beta	se	tstat	pval
20	1:753405:C:A	C	0.129036	False	359983	627065.0	17173900.0	0.000436	0.016685	0.026138	0.979147

REF = A2 =C
ALT= A1 = A


Plink individual data(chr1_eur.bim) NO HEADER

1       1:753405:C:A    0       753405  C       A

merged-data

1       1:753405:C:A    0       753405  C       A


#### For  1:692794:CA:C (flipped)

From the vcf file format 
 CHROM    POS       ID        REF     ALT     QUAL    FILTER  INFO    FORMAT 

From the vcf file 

1       692794        .       CA      C  

From the updated_vcf file 

1       692794  1:692794:CA:C   CA      C   

From row gwas 

SNPID	minor_allele	minor_AF	low_confidence_variant	n_complete_samples	AC	ytx	beta	se	tstat	pval	CHR	POS	A2	A1
4	1:692794:CA:C	C	0.11059	False	359983	79621.1	2179000.0	-0.016436	0.019585	-0.839228	0.401342	1	692794	CA	C

REF = A2 =CA
ALT= A1 = C


Plink individual data(chr1_eur.bim) NO HEADER

1       1:692794:CA:C   0       692794  C       CA

merged-data

1       1:692794:CA:C   0       692794  C       CA


In [45]:
gwas_data_df[gwas_data_df["POS"]==753405]

,SNPID,minor_allele,minor_AF,low_confidence_variant,n_complete_samples,AC,ytx,beta,se,tstat,pval,CHR,POS,A2,A1
20,1:753405:C:A,C,0.129036,False,359983,627065.0,17173900.0,0.000436,0.016685,0.026138,0.979147,1,753405,C,A


In [46]:
ref_bim[ref_bim["POS"]==753405]

,CHR,SNP,CM,POS,A2,A1
11,1,1:753405:C:A,0,753405,C,A


In [55]:
ref_bim[ref_bim["POS"]==692794]

,CHR,SNP,CM,POS,A2,A1
0,1,1:692794:CA:C,0,692794,C,CA


In [65]:
mismatch

0          True
1          True
2          True
3          True
4          True
           ... 
6633465    True
6633466    True
6633467    True
6633468    True
6633469    True
Length: 6633470, dtype: bool

In [66]:
swapped_mismatch = (
    (merged["A1_gwas"] == merged["A2_ref"]) & 
    (merged["A2_gwas"] == merged["A1_ref"])
)
print(f"Simple allele swaps: {swapped_mismatch.sum()}")

Simple allele swaps: 4915510


In [68]:
complement = {'A':'T', 'T':'A', 'C':'G', 'G':'C', 'D':'D', 'I':'I'}

flipped_mismatch = (
    (merged["A1_gwas"].map(complement) == merged["A1_ref"]) & 
    (merged["A2_gwas"].map(complement) == merged["A2_ref"])
)
print(f"Strand flipped SNPs: {flipped_mismatch.sum()}")

Strand flipped SNPs: 684354


In [69]:
def harmonize_alleles(row):
    if row["A1_gwas"] == row["A1_ref"] and row["A2_gwas"] == row["A2_ref"]:
        return 1, row["beta"]
    elif row["A1_gwas"] == row["A2_ref"] and row["A2_gwas"] == row["A1_ref"]:
        return 1, -row["beta"] 
    
    elif (complement.get(row["A1_gwas"], '') == row["A1_ref"] and 
          complement.get(row["A2_gwas"], '') == row["A2_ref"]):
        return 1, row["beta"]  
    
    elif (complement.get(row["A1_gwas"], '') == row["A2_ref"] and 
          complement.get(row["A2_gwas"], '') == row["A1_ref"]):
        return 1, -row["beta"]  
    
    else:
        return 0, np.nan  

merged[["keep", "harmonized_beta"]] = merged.apply(harmonize_alleles, axis=1, result_type="expand")

print(f"Successfully harmonized: {merged['keep'].sum()}")
print(f"Remaining mismatches: {len(merged) - merged['keep'].sum()}")

Successfully harmonized: 6622047.0
Remaining mismatches: 11423.0


In [71]:
merged.head()

,SNPID,minor_allele,minor_AF,low_confidence_variant,n_complete_samples,AC,ytx,beta,se,tstat,...,CHR,POS,A2_gwas,A1_gwas,SNP,CM,A2_ref,A1_ref,keep,harmonized_beta
0,1:692794:CA:C,C,0.110590,False,359983,79621.1,2179000.0,-0.016436,0.019585,-0.839228,...,1,692794,CA,C,1:692794:CA:C,0,C,CA,1.0,0.016436
1,1:693731:A:G,G,0.115767,False,359983,83348.0,2281760.0,-0.004255,0.018507,-0.229918,...,1,693731,A,G,1:693731:A:G,0,G,A,1.0,0.004255
2,1:707522:G:C,C,0.097255,False,359983,70020.6,1916410.0,-0.010428,0.020804,-0.501258,...,1,707522,G,C,1:707522:G:C,0,C,G,1.0,0.010428
3,1:730087:T:C,C,0.056424,False,359983,40623.1,1110650.0,-0.047388,0.025783,-1.837930,...,1,730087,T,C,1:730087:T:C,0,C,T,1.0,0.047388
4,1:731718:T:C,C,0.121670,False,359983,87598.4,2397990.0,-0.005842,0.017552,-0.332853,...,1,731718,T,C,1:731718:T:C,0,C,T,1.0,0.005842


In [72]:
harmonized_df = merged[merged["keep"] == 1].copy()
output_cols = ["CHR", "POS", "A1_ref", "A2_ref", "harmonized_beta", "se", "pval", "SNP", "minor_AF"]
harmonized_df = harmonized_df[output_cols]

harmonized_df.columns = ["CHR", "POS", "A1", "A2", "beta", "se", "pval", "SNP_ref", "minor_AF"]

harmonized_df.to_csv("../../data/susie/mismatch_test/harmonized_gwas_data.csv", index=False)

In [73]:
harmonized_df=harmonized_df.rename(columns={'SNP_ref':'SNPID'})
harmonized_df.head()

,CHR,POS,A1,A2,beta,se,pval,SNPID,minor_AF
0,1,692794,CA,C,0.016436,0.019585,0.401342,1:692794:CA:C,0.110590
1,1,693731,A,G,0.004255,0.018507,0.818155,1:693731:A:G,0.115767
2,1,707522,G,C,0.010428,0.020804,0.616190,1:707522:G:C,0.097255
3,1,730087,T,C,0.047388,0.025783,0.066074,1:730087:T:C,0.056424
4,1,731718,T,C,0.005842,0.017552,0.739246,1:731718:T:C,0.121670


In [74]:
harmonized_df["SNPID"].to_csv("../../data/susie/mismatch_test/harmonized_gwas_data_snps.txt", index=False)

In [75]:

print(f"Duplicate positions: {harmonized_df.duplicated(['CHR','POS']).sum()}")
print(harmonized_df["A1"].value_counts())
print(harmonized_df["A2"].value_counts())

Duplicate positions: 2264
A1
C                  1712844
G                  1672538
A                  1423851
T                  1418850
CT                   34077
                    ...   
CCATTGCACT               1
CTGCTTTAAATGTT           1
GCATGTGTGTGTGTA          1
GGTGCTT                  1
GGGCTGTGCTGA             1
Name: count, Length: 30344, dtype: int64
A2
A                     1665050
T                     1648178
C                     1509141
G                     1457333
CT                      34132
                       ...   
GTGTTGCTAAC                 1
TCCACCCACCCACCCA            1
AAAAAAGG                    1
AAAGCGAGTCTAGTCCAG          1
ATGTGATG                    1
Name: count, Length: 21636, dtype: int64


In [76]:
problem_variants = merged[merged["keep"] == 0]
print(problem_variants["A1_gwas"].str.len().value_counts())
print(problem_variants["A2_gwas"].str.len().value_counts())

harmonized_df = harmonized_df[harmonized_df["A1"].str.len() == 1]
harmonized_df = harmonized_df[harmonized_df["A2"].str.len() == 1]

A1_gwas
1     5957
2     1408
3     1197
5      737
4      443
9      384
7      303
13     225
6      172
11     116
10      93
17      72
8       55
16      54
21      51
19      24
12      21
14      19
15      19
18      15
22      15
25      12
20       7
26       7
27       5
28       2
35       2
23       1
45       1
31       1
33       1
42       1
24       1
55       1
49       1
Name: count, dtype: int64
A2_gwas
1     9590
3      480
2      332
5      264
4      150
9      117
6      106
7       94
13      62
11      46
8       37
10      29
17      18
16      15
12      14
14      13
15      12
19       9
21       5
18       4
20       3
27       3
23       2
24       2
29       2
25       2
26       1
30       1
68       1
28       1
37       1
44       1
50       1
22       1
38       1
45       1
33       1
52       1
Name: count, dtype: int64


In [77]:
harmonized_df = harmonized_df.drop_duplicates(["CHR", "POS"], keep="first")

In [78]:
print("Final allele distributions:")
print(harmonized_df["A1"].value_counts())
print(harmonized_df["A2"].value_counts())

harmonized_df["MAF"] = harmonized_df[["minor_AF"]].min(axis=1)
print("MAF distribution:")
print(harmonized_df["MAF"].describe())

Final allele distributions:
A1
G    1603660
C    1602883
A    1340777
T    1338491
Name: count, dtype: int64
A2
A    1564712
T    1552912
C    1384253
G    1383934
Name: count, dtype: int64
MAF distribution:
count    5.885811e+06
mean     2.370609e-01
std      1.339198e-01
min      1.011640e-02
25%      1.155030e-01
50%      2.172780e-01
75%      3.498780e-01
max      5.000000e-01
Name: MAF, dtype: float64


In [79]:
harmonized_df.head()

,CHR,POS,A1,A2,beta,se,pval,SNPID,minor_AF,MAF
1,1,693731,A,G,0.004255,0.018507,0.818155,1:693731:A:G,0.115767,0.115767
2,1,707522,G,C,0.010428,0.020804,0.616190,1:707522:G:C,0.097255,0.097255
3,1,730087,T,C,0.047388,0.025783,0.066074,1:730087:T:C,0.056424,0.056424
4,1,731718,T,C,0.005842,0.017552,0.739246,1:731718:T:C,0.121670,0.121670
5,1,732032,A,C,0.000208,0.018722,0.991130,1:732032:A:C,0.121110,0.121110


In [80]:
harmonized_df["N"] = 359983


In [81]:
harmonized_df.head()

,CHR,POS,A1,A2,beta,se,pval,SNPID,minor_AF,MAF,N
1,1,693731,A,G,0.004255,0.018507,0.818155,1:693731:A:G,0.115767,0.115767,359983
2,1,707522,G,C,0.010428,0.020804,0.616190,1:707522:G:C,0.097255,0.097255,359983
3,1,730087,T,C,0.047388,0.025783,0.066074,1:730087:T:C,0.056424,0.056424,359983
4,1,731718,T,C,0.005842,0.017552,0.739246,1:731718:T:C,0.121670,0.121670,359983
5,1,732032,A,C,0.000208,0.018722,0.991130,1:732032:A:C,0.121110,0.121110,359983


In [82]:
formatted_cojo_df = harmonized_df.rename(columns={
    'SNPID': 'SNP',
    'A1_ref': 'A1',
    'A2_ref': 'A2',
    'beta': 'b',
    'minor_AF':'freq',
    'se': 'se',
    'pval': 'p',
    'N':'N'
    
})

cojo_ready_df = formatted_cojo_df[['SNP', 'A1', 'A2', 'freq', 'b', 'se', 'p', 'N']]
cojo_ready_df.head()
cojo_ready_df.to_csv("../../data/susie/mismatch_test/cojo_extracted_file_37.csv", sep=" ", index=False)

cojo

icog-bioai@dl:/mnt/hdd_2/rediet/hypothesis-generation-demo/data/susie/gcta/gcta-1.94.3-linux-kernel-3-x86_64$ gcta64 --bfile ../../plink_binary/merged_data --cojo-file ../../mismatch_test/cojo_extracted_file_37.csv --cojo-slct --out all_data_chr_cojo

(cyvcf_env) icog-bioai@dl:/mnt/hdd_1/rediet/hypothesis-generation-demo/data/susie/gcta/gcta-1.94.3-linux-kernel-3-x86_64$ gcta64 --bfile ../../test_plink/plink_binary/chr16_eur --cojo-file ../../test_plink/cojo_extracted_file_37.csv --cojo-slct --out all_chr_cojo
*******************************************************************
* Genome-wide Complex Trait Analysis (GCTA)
* version v1.94.1 Linux
* Built at Dec 16 2024 16:35:13, by GCC 8.3
* (C) 2010-present, Yang Lab, Westlake University
* Please report bugs to Jian Yang <jian.yang@westlake.edu.cn>
*******************************************************************
Analysis started at 08:10:19 PDT on Sat May 31 2025.
Hostname: dl

Accepted options:
--bfile ../../test_plink/plink_binary/chr16_eur
--cojo-file ../../test_plink/cojo_extracted_file_37.csv
--cojo-slct
--out all_chr_cojo


Reading PLINK FAM file from [../../test_plink/plink_binary/chr16_eur.fam].
503 individuals to be included from [../../test_plink/plink_binary/chr16_eur.fam].
Reading PLINK BIM file from [../../test_plink/plink_binary/chr16_eur.bim].
2697949 SNPs to be included from [../../test_plink/plink_binary/chr16_eur.bim].
Warning: Duplicated SNP ID "16:12018309:T:<CN0>" has been changed to "16:12018309:T:<CN0>_522327"
.Reading PLINK BED file from [../../test_plink/plink_binary/chr16_eur.bed] in SNP-major format ...
Genotype data for 503 individuals and 2697949 SNPs to be included from [../../test_plink/plink_binary/chr16_eur.bed].

Reading GWAS summary-level statistics from [../../test_plink/cojo_extracted_file_37.csv] ...
GWAS summary statistics of 282186 SNPs read from [../../test_plink/cojo_extracted_file_37.csv].
Phenotypic variance estimated from summary statistics of all 282186 SNPs: 22.5069 (variance of logit for case-control studies).
Matching the GWAS meta-analysis results to the genotype data ...
Calculating allele frequencies ...
42 SNP(s) have large difference of allele frequency between the GWAS summary data and the reference sample. These SNPs have been saved in [all_chr_cojo.freq.badsnps].
282136 SNPs are matched to the genotype data.
Calculating the variance of SNP genotypes ...

Performing stepwise model selection on 282136 SNPs to select association signals ... (p cutoff = 5e-08; collinearity cutoff = 0.9)
(Assuming complete linkage equilibrium between SNPs which are more than 10Mb away from each other)
5 associated SNPs have been selected.
10 associated SNPs have been selected.
15 associated SNPs have been selected.
20 associated SNPs have been selected.
25 associated SNPs have been selected.
30 associated SNPs have been selected.
35 associated SNPs have been selected.
40 associated SNPs have been selected.
45 associated SNPs have been selected.
50 associated SNPs have been selected.
55 associated SNPs have been selected.
60 associated SNPs have been selected.
65 associated SNPs have been selected.
70 associated SNPs have been selected.
75 associated SNPs have been selected.
80 associated SNPs have been selected.
85 associated SNPs have been selected.
90 associated SNPs have been selected.
95 associated SNPs have been selected.
100 associated SNPs have been selected.
105 associated SNPs have been selected.
110 associated SNPs have been selected.
115 associated SNPs have been selected.
120 associated SNPs have been selected.
125 associated SNPs have been selected.
130 associated SNPs have been selected.
135 associated SNPs have been selected.
140 associated SNPs have been selected.
145 associated SNPs have been selected.
150 associated SNPs have been selected.
155 associated SNPs have been selected.
160 associated SNPs have been selected.
Finally, 161 associated SNPs are selected.
Performing joint analysis on all the 161 selected signals ...
Saving the 161 independent signals to [all_chr_cojo.jma.cojo] ...
Saving the LD structure of 161 independent signals to [all_chr_cojo.ldr.cojo] ...
Saving the conditional analysis results of 140933 remaining SNPs to [all_chr_cojo.cma.cojo] ...
(1 SNPs eliminated by backward selection and 141041 SNPs filtered by collinearity test are not included in the output)

Analysis finished at 08:27:49 PDT on Sat May 31 2025
Overall computational time: 17 minutes 30 sec.

In [ ]:
import pandas as pd
cojo_results_df = pd.read_csv("../../data/susie/gcta/gcta-1.94.3-linux-kernel-3-x86_64/all_chr_cojo.jma.cojo", delim_whitespace=True)
cojo_results_df.head()

/tmp/ipykernel_1673061/1798388032.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  cojo_results_df = pd.read_csv("../../data/susie/gcta/gcta-1.94.3-linux-kernel-3-x86_64/all_chr_cojo.jma.cojo", delim_whitespace=True)


,Chr,SNP,bp,refA,freq,b,se,p,n,freq_geno,bJ,bJ_se,pJ,LD_r
0,16,16:206865:A:C,206865,C,0.035055,0.079437,0.030358,8.878570e-03,360986,0.038767,0.23986,0.031948,6.009250e-14,0.043364
1,16,16:406427:G:C,406427,C,0.355430,-0.080681,0.011680,4.931170e-12,360007,0.310139,-1.70589,0.017352,0.000000e+00,-0.050285
2,16,16:629288:A:C,629288,C,0.038775,0.068293,0.029632,2.118110e-02,343868,0.028827,2.40642,0.035831,0.000000e+00,0.059069
3,16,16:831608:C:T,831608,T,0.011049,0.049277,0.055260,3.725350e-01,337264,0.004970,-3.48802,0.065008,0.000000e+00,0.128109
4,16,16:931605:G:C,931605,C,0.074670,-0.048957,0.021191,2.087260e-02,362684,0.073559,-2.34460,0.028327,0.000000e+00,0.028716


In [ ]:
most_significant_snp = cojo_results_df.sort_values(by='p').head(1)
most_significant_snp

,Chr,SNP,bp,refA,freq,b,se,p,n,freq_geno,bJ,bJ_se,pJ,LD_r
104,16,16:53802494:C:T,53802494,T,0.402018,0.359699,0.011353,2.721410e-220,362165,0.432406,3.43735,0.027826,0.0,0.133934


In [53]:
harmonized_df.head()

,CHR,POS,A1,A2,beta,se,pval,SNPID,minor_AF,MAF,N
1,1,693731,G,A,-0.004255,0.018507,0.818155,1:693731:A:G,0.115767,0.115767,359983
2,1,707522,C,G,-0.010428,0.020804,0.616190,1:707522:G:C,0.097255,0.097255,359983
3,1,730087,C,T,-0.047388,0.025783,0.066074,1:730087:T:C,0.056424,0.056424,359983
4,1,731718,C,T,-0.005842,0.017552,0.739246,1:731718:T:C,0.121670,0.121670,359983
5,1,732032,C,A,-0.000208,0.018722,0.991130,1:732032:A:C,0.121110,0.121110,359983


In [83]:
harmonized_df[harmonized_df["POS"]==53828066]

,CHR,POS,A1,A2,beta,se,pval,SNPID,minor_AF,MAF,N
5672664,16,53828066,C,T,-0.349002,0.011405,2.244600e-205,16:53828066:C:T,0.392165,0.392165,359983


In [84]:
variant_position = 53802494
start_pos = variant_position - 500000
end_pos = variant_position + 500000

df_region = harmonized_df[
        (harmonized_df["CHR"] == "16") & 
        (harmonized_df["POS"] >= start_pos) & 
        (harmonized_df["POS"] <= end_pos)
    ]
    
    


In [85]:
df_region.head()

,CHR,POS,A1,A2,beta,se,pval,SNPID,minor_AF,MAF,N
5671628,16,53304380,C,T,0.003661,0.012142,0.763024,16:53304380:C:T,0.305347,0.305347,359983
5671629,16,53304664,G,A,0.003678,0.012142,0.761946,16:53304664:G:A,0.305335,0.305335,359983
5671632,16,53308315,A,T,0.003084,0.012145,0.799574,16:53308315:A:T,0.305168,0.305168,359983
5671633,16,53308814,C,T,0.004453,0.018868,0.813407,16:53308814:C:T,0.096463,0.096463,359983
5671634,16,53308951,T,G,0.008033,0.012463,0.519181,16:53308951:T:G,0.279793,0.279793,359983


In [86]:
df_region=df_region.rename(columns={'SNP_ref':'SNPID'})
df_region.head()

,CHR,POS,A1,A2,beta,se,pval,SNPID,minor_AF,MAF,N
5671628,16,53304380,C,T,0.003661,0.012142,0.763024,16:53304380:C:T,0.305347,0.305347,359983
5671629,16,53304664,G,A,0.003678,0.012142,0.761946,16:53304664:G:A,0.305335,0.305335,359983
5671632,16,53308315,A,T,0.003084,0.012145,0.799574,16:53308315:A:T,0.305168,0.305168,359983
5671633,16,53308814,C,T,0.004453,0.018868,0.813407,16:53308814:C:T,0.096463,0.096463,359983
5671634,16,53308951,T,G,0.008033,0.012463,0.519181,16:53308951:T:G,0.279793,0.279793,359983


In [87]:
df_region["SNPID"].to_csv("../../data/susie/mismatch_test/harmonized_gwas_data_region.txt", index=False)

In [88]:
!plink \
  --bfile "../../data/susie/plink_binary/merged_data" \
  --keep-allele-order \
  --r square \
  --extract ../../data/susie/mismatch_test/harmonized_gwas_data_region.txt \
  --out ../../data/susie/mismatch_test/test_sig_locus_mt

PLINK v1.90b7.2 64-bit (11 Dec 2023)           www.cog-genomics.org/plink/1.9/
(C) 2005-2023 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to ../../data/susie/mismatch_test/test_sig_locus_mt.log.
Options in effect:
  --bfile ../../data/susie/plink_binary/merged_data
  --extract ../../data/susie/mismatch_test/harmonized_gwas_data_region.txt
  --keep-allele-order
  --out ../../data/susie/mismatch_test/test_sig_locus_mt
  --r square

257419 MB RAM detected; reserving 128709 MB for main workspace.
6622286 variants loaded from .bim file.
503 people (0 males, 0 females, 503 ambiguous) loaded from .fam.
Ambiguous sex IDs written to
../../data/susie/mismatch_test/test_sig_locus_mt.nosex .
--extract: 2056 variants remaining.
Using up to 27 threads (change this with --threads).
Before main variant filters, 503 founders and 0 nonfounders present.
Calculating allele frequencies... 1011121314151617181920212223242526272829303132333435363738394041424344454647484950515253545

In [89]:
import rpy2
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
import rpy2.robjects.numpy2ri as numpy2ri
numpy2ri.activate()

INFO:rpy2.situation:cffi mode is CFFI_MODE.ANY
INFO:rpy2.situation:R home found: /usr/lib/R
INFO:rpy2.situation:R library path: /usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/home/icog-bioai/jdk-11.0.2/lib/server
INFO:rpy2.situation:LD_LIBRARY_PATH: /usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/home/icog-bioai/jdk-11.0.2/lib/server
INFO:rpy2.rinterface_lib.embedded:Default options to initialize R: rpy2, --quiet, --no-save
INFO:rpy2.rinterface_lib.embedded:R is already initialized. No need to initialize.


In [90]:
susieR = importr('susieR')

In [91]:
ld_r = pd.read_csv("../../data/susie/mismatch_test/test_sig_locus_mt.ld", sep="\t", header=None)
R_df = ld_r.values


In [92]:
df_region.head()

,CHR,POS,A1,A2,beta,se,pval,SNPID,minor_AF,MAF,N
5671628,16,53304380,C,T,0.003661,0.012142,0.763024,16:53304380:C:T,0.305347,0.305347,359983
5671629,16,53304664,G,A,0.003678,0.012142,0.761946,16:53304664:G:A,0.305335,0.305335,359983
5671632,16,53308315,A,T,0.003084,0.012145,0.799574,16:53308315:A:T,0.305168,0.305168,359983
5671633,16,53308814,C,T,0.004453,0.018868,0.813407,16:53308814:C:T,0.096463,0.096463,359983
5671634,16,53308951,T,G,0.008033,0.012463,0.519181,16:53308951:T:G,0.279793,0.279793,359983


In [95]:
import rpy2.robjects as ro
ro.r('set.seed(123)')
fit = susieR.susie_rss(
    bhat = df_region["beta"].values.reshape(len(df_region), 1),
    shat =df_region["se"].values.reshape(len(df_region), 1),
    R = R_df,
    L = 1,
    n = 359983
)

credible_sets = susieR.susie_get_cs(fit, coverage=0.95, min_abs_corr=0.5, Xcorr=R_df)

In [96]:
credible_sets = susieR.susie_get_cs(fit, coverage=0.95, min_abs_corr=0.5, Xcorr=R_df)
print(credible_sets)

$cs
$cs$L1
[1] 799 803 806 807 814


$purity
   min.abs.corr mean.abs.corr median.abs.corr
L1     0.992977     0.9971908               1

$cs_index
[1] 1

$coverage
[1] 0.9705779

$requested_coverage
[1] 0.95




In [97]:
df_region.loc[:, "cs"] = 0

In [98]:

df_region.loc[:, "cs"] = 0
n_cs = len(susieR.susie_get_cs(fit, coverage = 0.95, min_abs_corr = 0.5, Xcorr = R_df)[0])
print(n_cs)
credible_sets = susieR.susie_get_cs(fit, coverage=0.95, min_abs_corr=0.5, Xcorr=R_df)[0]

for i in range(n_cs):
    cs_index = credible_sets[i]
    df_region.iloc[np.array(cs_index) - 1, df_region.columns.get_loc("cs")] = i + 1


df_region["pip"] = np.array(susieR.susie_get_pip(fit))

1


In [99]:
df_region = df_region.reset_index(drop=True)


In [100]:
cs_index = np.array(credible_sets[0])
for i in range(1, n_cs):
    cs_index = np.concatenate((cs_index, np.array(credible_sets[i])), axis=0)
credible_snps = df_region.loc[cs_index - 1, :]

print(credible_snps)

    CHR       POS A1 A2      beta        se           pval            SNPID  \
798  16  53800954  T  C -0.359304  0.011353  1.569760e-219  16:53800954:T:C   
802  16  53802494  C  T -0.359699  0.011353  5.465200e-220  16:53802494:C:T   
805  16  53803223  G  A -0.359563  0.011354  8.107390e-220  16:53803223:G:A   
806  16  53803574  T  A -0.359554  0.011353  8.187040e-220  16:53803574:T:A   
813  16  53806453  A  G -0.359082  0.011349  2.114840e-219  16:53806453:A:G   

     minor_AF       MAF       N  cs       pip  
798  0.401795  0.401795  359983   1  0.114891  
802  0.402018  0.402018  359983   1  0.328532  
805  0.401998  0.401998  359983   1  0.221432  
806  0.402013  0.402013  359983   1  0.219802  
813  0.403039  0.403039  359983   1  0.085920  


In [101]:
credible_snps_sorted = credible_snps.sort_values(by="pip", ascending=False)
print(credible_snps_sorted[["SNPID", "pip"]])

               SNPID       pip
802  16:53802494:C:T  0.328532
805  16:53803223:G:A  0.221432
806  16:53803574:T:A  0.219802
798  16:53800954:T:C  0.114891
813  16:53806453:A:G  0.085920


In [102]:
import requests


chrom = credible_snps["CHR"]
pos = credible_snps["POS"]
pip = credible_snps["pip"]
crediable_snps_rs = []

def get_rsid(chrom, pos, genome="hg19"):
    url = f"https://api.genome.ucsc.edu/getData/track?genome={genome};track=dbSnp155;chrom=chr{chrom};start={pos-1};end={pos}"
    response = requests.get(url).json()
    if "dbSnp155" in response and response["dbSnp155"]:
        return response["dbSnp155"][0]["name"]
    return None

for c, p, pip in zip(chrom, pos, pip):
    rsid = get_rsid(c, p)
    crediable_snps_rs.append([rsid, c, p, pip])

df_crediable_snps= pd.DataFrame(crediable_snps_rs, columns=["rsID", "Chromosome", "Position", "pip"])
df_crediable_snps = df_crediable_snps.sort_values(by="pip", ascending=False)


In [103]:
df_crediable_snps

,rsID,Chromosome,Position,pip
1,rs11642015,16,53802494,0.328532
2,rs62048402,16,53803223,0.221432
3,rs1558902,16,53803574,0.219802
0,rs1421085,16,53800954,0.114891
4,rs56094641,16,53806453,0.085920
